In [ ]:
import tensorflow as tf
import matplotlib.pyplot as plt
from tensorflow.keras.datasets import fashion_mnist
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Flatten, Dense

# Load Dataset
(x_train, y_train), (x_test, y_test) = fashion_mnist.load_data()

# Normalize
x_train = x_train / 255.0
x_test = x_test / 255.0

# Build Model
model = Sequential([
    Flatten(input_shape=(28, 28)),
    Dense(128, activation='relu'),
    Dense(64, activation='relu'),
    Dense(10, activation='softmax')
])

# Compile Model
model.compile(
    optimizer='adam',
    loss='sparse_categorical_crossentropy',
    metrics=['accuracy']
)

# Train Model
history = model.fit(
    x_train,
    y_train,
    epochs=10,
    validation_split=0.2
)

# Plot Loss Curves
plt.figure(figsize=(8,5))
plt.plot(history.history['loss'], label='Training Loss')
plt.plot(history.history['val_loss'], label='Validation Loss')
plt.xlabel("Epoch")
plt.ylabel("Loss")
plt.title("Training vs Validation Loss")
plt.legend()
plt.show()

Epoch 1/10
1500/1500 ━━━━━━━━━━━━━━━━━━━━ 12s 6ms/step - accuracy: 0.8185 - loss: 0.5102 - val_accuracy: 0.8507 - val_loss: 0.4077
Epoch 2/10
1500/1500 ━━━━━━━━━━━━━━━━━━━━ 9s 6ms/step - accuracy: 0.8618 - loss: 0.3782 - val_accuracy: 0.8643 - val_loss: 0.3777
Epoch 3/10
1493/1500 ━━━━━━━━━━━━━━━━━━━━ 0s 5ms/step - accuracy: 0.8753 - loss: 0.3417

In [ ]:
import matplotlib.pyplot as plt

# Use Only 500 Samples
x_small = x_train[:500]
y_small = y_train[:500]

# Large Model to Encourage Overfitting
overfit_model = Sequential([
    Flatten(input_shape=(28,28)),
    Dense(512, activation='relu'),
    Dense(256, activation='relu'),
    Dense(128, activation='relu'),
    Dense(10, activation='softmax')
])

overfit_model.compile(
    optimizer='adam',
    loss='sparse_categorical_crossentropy',
    metrics=['accuracy']
)

history_overfit = overfit_model.fit(
    x_small,
    y_small,
    epochs=30,
    validation_split=0.2
)

# Plot Loss Curves
plt.figure(figsize=(8,5))
plt.plot(history_overfit.history['loss'], label='Training Loss')
plt.plot(history_overfit.history['val_loss'], label='Validation Loss')
plt.xlabel("Epoch")
plt.ylabel("Loss")
plt.title("Overfitting Example")
plt.legend()
plt.show()

print("Signs of Overfitting:")
print("1. Training loss continues to decrease while validation loss starts increasing.")
print("2. The gap between training loss and validation loss becomes larger after several epochs.")

In [ ]:
from tensorflow.keras.callbacks import EarlyStopping

# EarlyStopping Callback
early_stop = EarlyStopping(
    monitor='val_loss',
    patience=3,
    restore_best_weights=True
)

# Train Model
history_es = model.fit(
    x_train,
    y_train,
    epochs=30,
    validation_split=0.2,
    callbacks=[early_stop]
)

print("Training stopped after",
      len(history_es.history['loss']),
      "epochs.")

In [ ]:
from sklearn.metrics import accuracy_score, precision_score, recall_score
import numpy as np

# Predictions
y_prob = model.predict(x_test)
y_pred = np.argmax(y_prob, axis=1)

# Metrics
accuracy = accuracy_score(y_test, y_pred)

precision = precision_score(
    y_test,
    y_pred,
    average='weighted'
)

recall = recall_score(
    y_test,
    y_pred,
    average='weighted'
)

print("Accuracy :", accuracy)
print("Precision:", precision)
print("Recall   :", recall)

In [ ]:
from sklearn.preprocessing import label_binarize
from sklearn.metrics import roc_curve, auc, roc_auc_score
import matplotlib.pyplot as plt

# Convert Labels to One-Hot
y_test_bin = label_binarize(y_test, classes=range(10))

# Predict Probabilities
y_score = model.predict(x_test)

# Compute Overall AUC
auc_score = roc_auc_score(
    y_test_bin,
    y_score,
    multi_class='ovr'
)

print("Overall AUC Score:", auc_score)

# ROC Curve for Class 0
fpr, tpr, _ = roc_curve(
    y_test_bin[:,0],
    y_score[:,0]
)

roc_auc = auc(fpr, tpr)

# Plot ROC Curve
plt.figure(figsize=(7,5))
plt.plot(fpr, tpr, label=f"AUC = {roc_auc:.3f}")
plt.plot([0,1],[0,1],'k--')
plt.xlabel("False Positive Rate")
plt.ylabel("True Positive Rate")
plt.title("ROC Curve (Class 0)")
plt.legend()
plt.show()